In [33]:
import sqlite3
import pandas as pd
import time
from datetime import datetime

## connection to our oltp database
conn=sqlite3.connect('unisole_rides.db')
cursor=conn.cursor()

print("OLTP database initialized")

OLTP database initialized


In [34]:
## creating tables with constarints
cursor.execute(''' 
CREATE TABLE IF NOT EXISTS rides(
   ride_id INTEGER PRIMARY KEY AUTOINCREMENT,
   user_id TEXT NOT NULL,
    driver_id TEXT,
    pickup_location TEXT,
    destination TEXT,
    status TEXT CHECK( status IN 
    ('requested','ongoing','compeletd','cancelled')),
    price REAL,
    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
''')
conn.commit()
print("table created")

table created


In [35]:
def book_ride(user_id,pickup,dest,price):
    try:
        #start Transaction
        cursor.execute("BEGIN TRANSACTION;")
         # STEP 1 INSERT RIDE 
        cursor.execute(''' 
             INSERT INTO rides(
                       user_id,
                       pickup_location,
                       destination,
                       status,
                       price
        )
        VALUES(?,?,?,'requested',?)
        ''',(user_id,pickup,dest,price))

       ## simulte payment
        payment_success = True
        if  not payment_success:
            raise Exception("payment failed")
        # save changes 
        conn.commit()
        print(f"ride book successfully for {user_id}")

    except Exception as e:
        # rollaback if anythings fails
        conn.rollback()
        print(f"transaction failed due to {e} . No data was saved ")

In [36]:
book_ride("nikita_102","mandi","hamirpur","45")

ride book successfully for nikita_102


In [37]:
def assign_driver(ride_id,driver_id):
    cursor.execute(
        "SELECT status FROM rides WHERE ride_id=?",(ride_id,)
    )
    status= cursor.fetchone()[0]

    if status == 'requested':
        cursor.execute(
            '''
            UPDATE rides 
            SET driver_id=?, status='ongoing'
            WHERE ride_id =?
        ''',
        (driver_id,ride_id)
        )

        conn.commit()
        print(f"Driver {driver_id} assigned to ride {ride_id}")
    else :
        print("ride already assigned or cancelled")

In [38]:
assign_driver(1,"driver_deepak")

ride already assigned or cancelled


In [39]:
def get_features_model(user_id):
    query = f"""
    SELECT AVG(price)
    FROM rides
    WHERE user_id = '{user_id}'
    AND status = 'completed'
    LIMIT 5
  """
    df = pd.read_sql_query(query,conn)
    return df.iloc[0,0]